# 🎬 Movie Recommendation System

## 📌 Overview

This project builds a movie recommendation system that suggests similar movies based on content/features. The goal is to help users discover movies they might like based on their preferences.

## 🛠️ Libraries Used

* **NumPy & Pandas** → data manipulation
* **NLTK** → text preprocessing (stopwords, lemmatization)
* **Scikit-learn** → TF-IDF vectorization and similarity computation


In [1]:
# Importing required libraries
import numpy as np
import pandas as pd

# Ignore warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

## 📂 Dataset Loading

In this step, we load the movie metadata dataset which contains details like movie titles, genres, and descriptions. This dataset will be used to build the recommendation system.


In [2]:
# Load the dataset
df = pd.read_csv('movies_metadata.csv')

### 🔍 Dataset Loaded

The dataset has been successfully loaded into a DataFrame and is ready for further exploration and preprocessing.


In [3]:
# Displaying the first 5 rows of the dataset to understand its structure
df.head()

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0
3,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",...,1995-12-22,81452156.0,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,...,1995-02-10,76578911.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0


### 🔍 Data Preview

We display the first few rows of the dataset to get an initial understanding of the data, including the available columns and their values.


In [4]:
# Listing all column names in the dataset
df.columns

Index(['adult', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id',
       'imdb_id', 'original_language', 'original_title', 'overview',
       'popularity', 'poster_path', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'video',
       'vote_average', 'vote_count'],
      dtype='object')

### 🧾 Column Names

This step helps us understand all the available features in the dataset, which is useful for selecting relevant columns for building the recommendation system.


In [5]:
# Displaying information about the dataset (data types, non-null values, memory usage)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45466 entries, 0 to 45465
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  45466 non-null  object 
 1   belongs_to_collection  4494 non-null   object 
 2   budget                 45466 non-null  object 
 3   genres                 45466 non-null  object 
 4   homepage               7782 non-null   object 
 5   id                     45466 non-null  object 
 6   imdb_id                45449 non-null  object 
 7   original_language      45455 non-null  object 
 8   original_title         45466 non-null  object 
 9   overview               44512 non-null  object 
 10  popularity             45461 non-null  object 
 11  poster_path            45080 non-null  object 
 12  production_companies   45463 non-null  object 
 13  production_countries   45463 non-null  object 
 14  release_date           45379 non-null  object 
 15  re

### 📊 Dataset Information

This provides a summary of the dataset, including:

* Data types of each column
* Number of non-null values
* Helps identify missing data and irrelevant features

This step is useful for planning data preprocessing.


In [6]:
# Displaying the number of rows and columns in the dataset
df.shape

(45466, 24)

### 📐 Dataset Size

This shows the dimensions of the dataset in terms of rows and columns. It helps in understanding the scale of the data we are working with.


## 🧹 Missing Value Analysis

Before building the recommendation system, it is important to identify missing values in the dataset. Missing data can affect the quality of recommendations, so we need to understand where they occur.


In [7]:
# Checking the number of missing values in each column
df.isnull().sum()

,0
adult,0
belongs_to_collection,40972
budget,0
genres,0
homepage,37684
id,0
imdb_id,17
original_language,11
original_title,0
overview,954


### ⚠️ Missing Values Summary

This shows the count of missing values in each column. Columns with a high number of missing values may need to be cleaned, modified, or removed during preprocessing.


In [8]:
# Checking for duplicate rows in the dataset
df.duplicated().sum()

np.int64(13)

### 🔁 Duplicate Records

This step checks whether there are any duplicate rows in the dataset. Duplicate entries can affect the recommendation results and may need to be removed during preprocessing.


In [9]:
# Removing duplicate rows and resetting the index
df = df.drop_duplicates().reset_index(drop=True)

### 🧼 Removing Duplicates

Duplicate rows are removed to ensure data quality. The index is reset after removal to maintain a clean and consistent DataFrame structure.


In [10]:
# Verifying that duplicate rows have been removed
df.duplicated().sum()

np.int64(0)

### ✅ Verification

After removing duplicates, we check again to confirm that no duplicate rows remain in the dataset.


## 🎯 Feature Selection

Not all columns in the dataset are useful for building a recommendation system. In this step, we select only the relevant features that will help in generating meaningful recommendations.


In [11]:
# Selecting relevant columns required for building the recommendation system
df = df[['title', 'overview', 'genres', 'tagline', 'vote_average', 'popularity']]

In [12]:
# Displaying the dataset after selecting relevant columns
df

,title,overview,genres,tagline,vote_average,popularity
0,Toy Story,"Led by Woody, Andy's toys live happily in his ...","[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",NaN,7.7,21.946943
1,Jumanji,When siblings Judy and Peter discover an encha...,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",Roll the dice and unleash the excitement!,6.9,17.015539
2,Grumpier Old Men,A family wedding reignites the ancient feud be...,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",Still Yelling. Still Fighting. Still Ready for...,6.5,11.7129
3,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...","[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",Friends are the people who let you be yourself...,6.1,3.859495
4,Father of the Bride Part II,Just when George Banks has recovered from his ...,"[{'id': 35, 'name': 'Comedy'}]",Just When His World Is Back To Normal... He's ...,5.7,8.387519
...,...,...,...,...,...,...
45448,Subdue,Rising and falling between a man and woman.,"[{'id': 18, 'name': 'Drama'}, {'id': 10751, 'n...",Rising and falling between a man and woman,4.0,0.072051
45449,Century of Birthing,An artist struggles to finish his work while a...,"[{'id': 18, 'name': 'Drama'}]",NaN,9.0,0.178241
45450,Betrayal,"When one of her hits goes wrong, a professiona...","[{'id': 28, 'name': 'Action'}, {'id': 18, 'nam...",A deadly game of wits.,3.8,0.903007
45451,Satan Triumphant,"In a small town live two brothers, one a minis...",[],NaN,0.0,0.003503


### 🔍 Dataset After Feature Selection

We display the dataset after selecting the relevant columns to verify that only the required features are retained for building the recommendation system.


### 📌 Selected Features

The selected columns include:

* **Title**: Name of the movie
* **Overview**: Short description of the movie
* **Genres**: Categories the movie belongs to
* **Tagline**: Additional descriptive text
* **Vote Average**: User rating
* **Popularity**: Popularity score

These features will be used to compute similarity between movies.


In [13]:
# Checking missing values again after selecting relevant columns
df.isnull().sum()

,0
title,6
overview,954
genres,0
tagline,25045
vote_average,6
popularity,5


### ⚠️ Missing Values After Feature Selection

After narrowing down the dataset to relevant columns, we recheck for missing values to ensure the selected features are clean and ready for further processing.


In [14]:
# Removing rows where the 'title' is missing, as it is essential for recommendations
df = df.dropna(subset=['title'])

### 🧹 Handling Missing Titles

Rows with missing movie titles are removed since the title is a key identifier in the recommendation system and cannot be left undefined.


In [15]:
# Replacing missing values in 'overview' with empty strings to avoid issues during text processing
df['overview'] = df['overview'].fillna('')

### 📝 Handling Missing Overviews

Missing values in the **overview** column are replaced with empty strings. Since this field is used for text-based similarity, filling missing values ensures smooth processing without losing data.


## 🔍 Understanding Genre Data

Before processing the genres column, it is important to inspect its structure. This helps in determining how to extract meaningful information for the recommendation system.


In [16]:
# Inspecting the format of the 'genres' column for a single row
df.iloc[0]['genres']

"[{'id': 16, 'name': 'Animation'}, {'id': 35, 'name': 'Comedy'}, {'id': 10751, 'name': 'Family'}]"

### 🧾 Genre Format

The genres column is not in a simple format. It typically contains structured data (like dictionaries or JSON-like strings), which needs to be processed before it can be used for analysis.


## 🔄 Genre Data Preprocessing

The genres column contains structured data in the form of JSON-like strings. To make it usable for the recommendation system, we extract only the genre names and convert them into a simple text format.


In [17]:
# Importing ast to safely evaluate string representations of Python objects
import ast

# Converting the 'genres' column from JSON-like strings to a space-separated string of genre names
df['genres'] = df['genres'].apply(
    lambda x: " ".join([i['name'] for i in ast.literal_eval(x)]) if x.startswith('[') else x
)

### 🏷️ Extracted Genres

The genres are now transformed into a clean, space-separated string format. This makes it easier to use them in text-based similarity calculations.


In [18]:
# Displaying the updated dataset after genre preprocessing
df.head()

,title,overview,genres,tagline,vote_average,popularity
0,Toy Story,"Led by Woody, Andy's toys live happily in his ...",Animation Comedy Family,NaN,7.7,21.946943
1,Jumanji,When siblings Judy and Peter discover an encha...,Adventure Fantasy Family,Roll the dice and unleash the excitement!,6.9,17.015539
2,Grumpier Old Men,A family wedding reignites the ancient feud be...,Romance Comedy,Still Yelling. Still Fighting. Still Ready for...,6.5,11.7129
3,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",Comedy Drama Romance,Friends are the people who let you be yourself...,6.1,3.859495
4,Father of the Bride Part II,Just when George Banks has recovered from his ...,Comedy,Just When His World Is Back To Normal... He's ...,5.7,8.387519


### 🔍 Updated Data Preview

We preview the dataset again to verify that the genres column has been successfully transformed into a clean, readable format.


In [19]:
# Replacing missing values in 'tagline' with empty strings
df['tagline'] = df['tagline'].fillna('')

### 📝 Handling Missing Taglines

Missing values in the **tagline** column are replaced with empty strings. This ensures consistency and avoids issues when combining text features later.


In [20]:
# Verifying missing values after handling the 'tagline' column
df.isnull().sum()

,0
title,0
overview,0
genres,0
tagline,0
vote_average,0
popularity,0


### ✅ Missing Values Check

After handling missing values in the **tagline** column, we recheck the dataset to confirm that the issue has been resolved and no unexpected missing values remain.


## 🧩 Feature Engineering

To build a recommendation system, we need a unified representation of each movie. In this step, we combine multiple text-based features into a single column.


In [21]:
# Combining overview, genres, and tagline into a single 'tags' column for similarity computation
df['tags'] = df['overview'] + ' ' + df['genres'] + ' ' + df['tagline']

### 🏷️ Creating Tags

The **tags** column is created by merging:

* Overview
* Genres
* Tagline

This combined text will be used to compute similarity between movies and generate recommendations.


In [22]:
# Displaying the dataset after creating the 'tags' column
df.head()

,title,overview,genres,tagline,vote_average,popularity,tags
0,Toy Story,"Led by Woody, Andy's toys live happily in his ...",Animation Comedy Family,,7.7,21.946943,"Led by Woody, Andy's toys live happily in his ..."
1,Jumanji,When siblings Judy and Peter discover an encha...,Adventure Fantasy Family,Roll the dice and unleash the excitement!,6.9,17.015539,When siblings Judy and Peter discover an encha...
2,Grumpier Old Men,A family wedding reignites the ancient feud be...,Romance Comedy,Still Yelling. Still Fighting. Still Ready for...,6.5,11.7129,A family wedding reignites the ancient feud be...
3,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",Comedy Drama Romance,Friends are the people who let you be yourself...,6.1,3.859495,"Cheated on, mistreated and stepped on, the wom..."
4,Father of the Bride Part II,Just when George Banks has recovered from his ...,Comedy,Just When His World Is Back To Normal... He's ...,5.7,8.387519,Just when George Banks has recovered from his ...


### 🔍 Updated Dataset Preview

We preview the dataset to verify that the **tags** column has been successfully created and contains the combined text features.


In [23]:
# Displaying the combined 'tags' text for the first movie
df['tags'][0]

"Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto the scene. Afraid of losing his place in Andy's heart, Woody plots against Buzz. But when circumstances separate Buzz and Woody from their owner, the duo eventually learns to put aside their differences. Animation Comedy Family "

### 🔍 Tags Preview

We inspect the **tags** column to verify that all text features have been combined correctly. This helps ensure the data is ready for similarity-based processing.


## 🧹 Text Preprocessing

To improve the quality of recommendations, the text data needs to be cleaned and standardized. This includes removing unnecessary words, normalizing text, and preparing it for similarity computation.


In [24]:
# Importing libraries for text preprocessing
import nltk
from nltk.corpus import stopwords  # for removing common words
from nltk.stem import WordNetLemmatizer  # for reducing words to base form
import re  # for text cleaning using regular expressions

### 🔧 Preprocessing Tools

We use:

* **NLTK** for natural language processing tasks
* **Stopwords** to remove common words that do not add much meaning
* **Lemmatization** to reduce words to their base form
* **Regular expressions** for cleaning text

These tools help transform raw text into a more meaningful representation.


In [25]:
# Downloading required NLTK resources for text preprocessing
nltk.download('stopwords')   # for removing common words
nltk.download('wordnet')     # for lemmatization

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

### ⬇️ Downloading Resources

The required datasets for stopword removal and lemmatization are downloaded using NLTK. These resources are essential for effective text preprocessing.


In [26]:
# Creating a set of English stopwords for filtering common words
stop_words = set(stopwords.words('english'))

# Initializing the lemmatizer to reduce words to their base form
lemmatizer = WordNetLemmatizer()

### ⚙️ Initializing Preprocessing Tools

We initialize:

* A set of **stopwords** to remove commonly used words that do not add much meaning
* A **lemmatizer** to convert words into their base form

These will be used to clean and standardize the text data.


## 🧼 Text Cleaning Function

To standardize the text data, we define a preprocessing function that performs multiple cleaning steps such as lowercasing, removing unwanted characters, eliminating stopwords, and applying lemmatization.


In [27]:
# Function to clean and preprocess text data
def preprocess_text(text):
    # Convert text to lowercase
    text = str(text).lower()

    # Remove special characters and punctuation
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)

    # Split text into individual words
    words = text.split()

    # Remove stopwords
    words = [word for word in words if word not in stop_words]

    # Apply lemmatization to reduce words to base form
    words = [lemmatizer.lemmatize(word) for word in words]

    # Join words back into a single string
    return " ".join(words)

### 🔍 Preprocessing Steps

The function performs the following operations:

* Converts text to lowercase
* Removes special characters and punctuation
* Splits text into words
* Removes stopwords
* Applies lemmatization

This ensures that the text data is clean and consistent for similarity calculations.


In [28]:
# Applying the preprocessing function to clean the 'tags' column
df['tags'] = df['tags'].apply(preprocess_text)

### 🧹 Applying Text Preprocessing

The preprocessing function is applied to the **tags** column to clean and standardize the text data. This step ensures that the text is ready for accurate similarity computation.


In [29]:
# Displaying the cleaned 'tags' text for the first movie after preprocessing
df['tags'][0]

'led woody andys toy live happily room andys birthday brings buzz lightyear onto scene afraid losing place andys heart woody plot buzz circumstance separate buzz woody owner duo eventually learns put aside difference animation comedy family'

### 🔍 Cleaned Tags Preview

We inspect the processed **tags** column to verify that text cleaning has been applied correctly, including removal of stopwords, punctuation, and normalization of words.


In [30]:
# Resetting the index after preprocessing steps to maintain a clean sequence
df = df.reset_index(drop=True)

### 🔄 Resetting Index

After multiple preprocessing steps, the index is reset to ensure a clean and continuous ordering of rows in the dataset.


## 🔗 Index Mapping

To efficiently retrieve movies during recommendation, we create a mapping between movie titles and their corresponding indices in the dataset.


In [31]:
# Creating a mapping from movie titles to their corresponding index
indices = pd.Series(df.index, index=df['title']).drop_duplicates()

# Displaying the mapping
indices

,0
title,
Toy Story,0
Jumanji,1
Grumpier Old Men,2
Waiting to Exhale,3
Father of the Bride Part II,4
...,...
Subdue,45442
Century of Birthing,45443
Betrayal,45444


### 🎯 Title-to-Index Mapping

This mapping allows us to quickly find the index of a movie using its title, which is essential for computing and retrieving recommendations.


## 🔢 Text Vectorization

To compute similarity between movies, we need to convert textual data into numerical form. This is done using the TF-IDF (Term Frequency–Inverse Document Frequency) technique.


In [32]:
# Importing TF-IDF Vectorizer to convert text data into numerical features
from sklearn.feature_extraction.text import TfidfVectorizer

### 📐 TF-IDF Representation

TF-IDF transforms text into vectors based on the importance of words. Words that appear frequently in a movie description but are rare across all movies are given higher importance.


In [33]:
# Initializing TF-IDF vectorizer with unigrams and bigrams, limiting features, and removing English stopwords
tfidf = TfidfVectorizer(max_features=50000, ngram_range=(1,2), stop_words='english')

### ⚙️ Vectorizer Configuration

The TF-IDF vectorizer is configured with:

* **max_features = 50000** → limits the number of features
* **ngram_range = (1, 2)** → considers both single words and word pairs
* **stop_words = 'english'** → removes common English words

This helps capture meaningful patterns from the text while controlling dimensionality.


In [34]:
# Fitting the TF-IDF vectorizer and transforming the 'tags' column into a matrix
tfidf_matrix = tfidf.fit_transform(df['tags'])

### 🔢 TF-IDF Matrix

The text data in the **tags** column is converted into a TF-IDF matrix. Each movie is now represented as a numerical vector based on the importance of words in its description.

In [35]:
# Displaying the TF-IDF matrix representation
tfidf_matrix

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 1555864 stored elements and shape (45447, 50000)>

### 🔍 Matrix Overview

This shows the TF-IDF matrix, where each row represents a movie and each column represents a feature (word or phrase). The values indicate the importance of each term in the movie's description.


## 📏 Similarity Computation

To recommend similar movies, we need a way to measure how closely related two movies are. This is done using cosine similarity on the TF-IDF vectors.


In [36]:
# Importing cosine similarity to measure similarity between movie vectors
from sklearn.metrics.pairwise import cosine_similarity

### 📐 Cosine Similarity

Cosine similarity measures the similarity between two vectors based on the angle between them. A higher value indicates that the movies are more similar in terms of their content.


## 🎯 Recommendation Function

We define a function that takes a movie title as input and returns a list of similar movies based on cosine similarity of their TF-IDF representations.


In [37]:
# Function to recommend similar movies based on a given title
def recommend(title, n=10):
    # Check if the movie exists in the dataset
    if title not in indices:
        return ['Movie not found']

    # Get the index of the given movie
    idx = indices[title]

    # Compute cosine similarity between the selected movie and all others
    sim_score = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).flatten()

    # Get indices of top 'n' most similar movies (excluding the movie itself)
    similar_idx = sim_score.argsort()[::-1][1:n+1]

    # Return the titles of the recommended movies
    return df['title'].iloc[similar_idx]

### 🔁 How It Works

* Finds the index of the given movie
* Computes similarity scores with all other movies
* Sorts movies based on similarity
* Returns the top **N** most similar movies

This forms the core logic of the recommendation system.


## 🎬 Sample Recommendation

We test the recommendation system by providing a movie title and retrieving similar movies based on content similarity.


In [38]:
# Getting movie recommendations for a sample input
recommend('La La Land')

,title
4231,There's No Business Like Show Business
595,A Great Day in Harlem
18621,Guy and Madeline on a Park Bench
23707,If I Were You
30227,Hier kommt Lola
30473,Intangible Asset Number 82
37685,Born to Be Blue
40570,Nina
8495,A Star Is Born
829,Dingo


### ✅ Output

The system returns a list of movies that are most similar to the given input movie based on textual features like overview, genres, and tagline.


## 📌 Conclusion

In this project, we built a content-based movie recommendation system using textual features such as overview, genres, and tagline.

The system:

* Cleans and preprocesses text data
* Converts text into numerical form using TF-IDF
* Uses cosine similarity to recommend similar movies

This approach can be further improved by incorporating user preferences, collaborative filtering, or deep learning techniques.
